# 07.05_Scanpy_AureliaSC_MDIC3_Python

单细胞 GRNBoost2 / MDIC3。

- 当前文件：`analysis/07_spatial_analysis/07.05_Scanpy_AureliaSC_MDIC3_Python.ipynb`
- 原始来源：`Codes/07.05_Scanpy_AureliaSC_MDIC3.ipynb`（旧编号仅用于溯源）。
- 运行内核：**python**。
- 导入依赖：`MDIC3`, `arboreto.algo`, `arboreto.utils`, `matplotlib.pyplot`, `networkx`, `numpy`, `os`, `pandas`, `scipy`, `seaborn`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。

**本文件说明：** 原 kernel 误标为 R，已依据 Python 正文修正元数据；代码保持原样。


In [ ]:
fig_dir = '/share/home/zhangze/zz/NeuralOrigin/Figures'

## GRNBoost2: Gene Regulatory Networks

### 1.1 GRNBoost for network

In [ ]:
import pandas as pd
import numpy as np
from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2

if __name__ == '__main__':
    # ex_matrix is a DataFrame with gene names as column names
    # ex_matrix = pd.read_csv(<ex_path>, sep='\t')
    exp = pd.read_csv("/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_MDIC3/Auco_SC.exp.txt", sep="\t", index_col=0)
    # transpose GRNBoost format：cells × genes
    ex_matrix = exp.T

    # tf_names is read using a utility function included in Arboreto
    # tf_names = load_tf_names(<tf_path>)
    tf_names = list(ex_matrix.columns)

    network = grnboost2(expression_data=ex_matrix,
                        tf_names=tf_names)

    # network.to_csv('output.tsv', sep='\t', index=False, header=False)
    print(network.shape)

In [ ]:
# type(network)
# pandas.core.frame.DataFrame
network.head()

### 1.2 network for GRN 

In [ ]:
# gene sort for MDIC3
genes = exp.index.tolist()
n_genes = len(genes)
# gene -> index mapping
gene2idx = {g: i for i, g in enumerate(genes)}
print("Number of genes:", n_genes)

# network: DataFrame with columns ['TF', 'target', 'importance']
GRN = np.zeros((n_genes, n_genes), dtype=np.float32)
miss_tf = 0
miss_tg = 0

for tf, target, weight in network[["TF", "target", "importance"]].itertuples(index=False):
    if tf not in gene2idx:
        miss_tf += 1
        continue
    if target not in gene2idx:
        miss_tg += 1
        continue

    i = gene2idx[tf]
    j = gene2idx[target]

    GRN[i, j] = weight

print("TF not found in exp genes:", miss_tf)
print("Target not found in exp genes:", miss_tg)

GRN

In [ ]:
np.savetxt("/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_MDIC3/Auco_SC.GRN.txt", GRN, fmt="%.6e")

## MDIC3: Cell-Cell Communication

### 2.1 MDIC3 for ccc_adjacency and type_adjacency

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy, statsmodels
print("scipy:", scipy.__version__)
print("statsmodels:", statsmodels.__version__)

In [ ]:
from MDIC3 import lucky
if __name__ == '__main__':

    AA, gene_exp, cellname = lucky.readexp('/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_MDIC3/Auco_SC.exp.txt')
    labels, label_index, label_cell = lucky.readlabel('/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_MDIC3/Auco_SC.metadata.txt')

    # import the GRN calculated by other methods or tools
    GRN = np.loadtxt('/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_MDIC3/Auco_SC.GRN.txt')

    old_cwd = os.getcwd()
    os.chdir('/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_MDIC3/Auco_SC')

    try:
        # Infer the cell-cell communication
        ccc_adjacency, type_adjacency = lucky.MDIC3_score(AA, GRN, labels, label_index)
        # User can save the inferred results using the following command
        lucky.MDIC3_scoresave(ccc_adjacency, type_adjacency, labels)
    finally:
        # Restore original working directory
        os.chdir(old_cwd) 

In [ ]:
ccc_adjacency

In [ ]:
type_adjacency

### 2.2 ccc_adjacency for filtering edges

In [ ]:
import numpy as np
import pandas as pd

ccc_adjacency = np.loadtxt("/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_MDIC3/Auco_SC/cellular_communication.txt")
print(ccc_adjacency.shape)
ccc_adjacency

In [ ]:
# all non-zero edges
rows, cols = np.nonzero(ccc_adjacency)
weights = ccc_adjacency[rows, cols]

edge_df = pd.DataFrame({
    "source": rows,
    "target": cols,
    "weight": weights,
    "abs_weight": np.abs(weights)
})
print("Number of all edges:", edge_df.shape[0])

# |weight| descent, top 2000
edge_df = (
    edge_df
    .sort_values("abs_weight", ascending=False)
    .head(2000)
    .drop(columns="abs_weight")
    .reset_index(drop=True)
)

print("Number of edges selected:", edge_df.shape[0])
edge_df

In [ ]:
# read metadata.txt
meta = pd.read_csv(
    "/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_MDIC3/Auco_SC.metadata.txt",
    sep="\t",
    header=None,
    names=["spot", "cluster_annos"]
)

# spot sort = ccc_adjacency sort
meta["spot_index"] = np.arange(meta.shape[0])
node_df = meta[["spot_index", "spot", "cluster_annos"]]
node_df


In [ ]:
# add source / target typese for dge_df
edge_df = edge_df.merge(
    node_df[["spot_index", "cluster_annos"]],
    left_on="source",
    right_on="spot_index",
    how="left"
).rename(columns={"cluster_annos": "source_type"}).drop(columns="spot_index")

edge_df = edge_df.merge(
    node_df[["spot_index", "cluster_annos"]],
    left_on="target",
    right_on="spot_index",
    how="left"
).rename(columns={"cluster_annos": "target_type"}).drop(columns="spot_index")

edge_df

In [ ]:
edge_df.to_csv("/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_MDIC3/Auco_SC.MDIC3.cccTop2000edges.csv", index=None)

## Auco SC Cell-cell communication Network

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

# ===============================
# 1. Load edge list
# ===============================
edge_df = pd.read_csv("/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_MDIC3/Auco_SC.MDIC3.cccTop2000edges.csv")

# ===============================
# 2. Derive node attributes
# ===============================
src_nodes = edge_df[["source", "source_type"]] \
    .rename(columns={"source": "spot", "source_type": "celltype"})
tgt_nodes = edge_df[["target", "target_type"]] \
    .rename(columns={"target": "spot", "target_type": "celltype"})

node_attr = pd.concat([src_nodes, tgt_nodes]).drop_duplicates()

# ===============================
# 3. Build directed graph
# ===============================
G = nx.DiGraph()

for _, r in node_attr.iterrows():
    G.add_node(r["spot"], celltype=r["celltype"])

for _, r in edge_df.iterrows():
    G.add_edge(r["source"], r["target"], weight=r["weight"])

# ===============================
# 4. Define cluster centers (by cell type)
# ===============================
celltypes = node_attr["celltype"].unique()
n_ct = len(celltypes)

angles = np.linspace(0, 2*np.pi, n_ct, endpoint=False)
centers = {
    ct: (10*np.cos(a), 10*np.sin(a))
    for ct, a in zip(celltypes, angles)
}

# ===============================
# 5. Assign node positions
# ===============================
rng = np.random.default_rng(0)

pos = {
    n: (
        centers[G.nodes[n]["celltype"]][0] + rng.normal(scale=0.6),
        centers[G.nodes[n]["celltype"]][1] + rng.normal(scale=0.6)
    )
    for n in G.nodes
}

# ===============================
# 6. Color by cell type
# ===============================
# palette = sns.color_palette("tab10", n_ct)
# ct_color = dict(zip(celltypes, palette))
# node_colors = [ct_color[G.nodes[n]["celltype"]] for n in G.nodes]
celltype_color_map = {
    'CN, Cnidocytes/Nematocyte cell': '#17becf', # CN 青色 '#17becf'
    'EM, Epidermal/Muscle cell': '#9467bd',      # EM 紫色 '#9467bd'
    'GA, Gastrodermal cell': '#ff7f0e',          # GA 橙色 '#ff7f0e'
    'GL, Gland cell': '#d62728',                 # GL 红色 '#d62728'
    'HA, Hair cell': '#8c564b',                  # HA 棕色 '#8c564b'
    'NE, Neural cell': '#2ca02c',                # NE 绿色 '#2ca02c'
    'SG, Stem/Germline cell': '#fedb61'          # SG 黄色 '#fedb61'
}
node_colors = [
    celltype_color_map.get(
        G.nodes[n]["celltype"],
        "#bbbbbb"   # 如果出现未定义的 cell type，给一个灰色兜底
    )
    for n in G.nodes
]

# ===============================
# 7. Edge visual attributes
# ===============================

# 取所有边的权重
edge_weights = np.array([d["weight"] for _, _, d in G.edges(data=True)])

# 线宽：按 |weight| 归一化到 [0.2, 3.0]
w_abs = np.abs(edge_weights)
w_min, w_max = w_abs.min(), w_abs.max()

edge_widths = 0.2 + 2.8 * (w_abs - w_min) / (w_max - w_min + 1e-8)

# 颜色：正 = 红，负 = 蓝
edge_colors = [
    # "#7F1D1D" if d["weight"] > 0 else "#1E3A8A"
    # "#C0392B" if d["weight"] > 0 else "#2980B9"
    "#F4A6A6" if d["weight"] > 0 else "#A7C7E7"
    for _, _, d in G.edges(data=True)
]

# ===============================
# 7. Plot
# ===============================
plt.figure(figsize=(7, 7))

nx.draw_networkx_edges(
    G,
    pos,
    edge_color=edge_colors,
    width=edge_widths,
    alpha=0.25,
    arrows=True,
    arrowstyle="-|>",
    arrowsize=8,
    connectionstyle="arc3,rad=0.1"  # 轻微弯曲，减少重叠
)

nx.draw_networkx_nodes(
    G, pos,
    node_size=20,
    node_color=node_colors,
    alpha=0.9
)

# for ct, (x, y) in centers.items():
#     plt.text(x, y, ct, fontsize=10,
#             #  fontweight="bold",
#              ha="center", va="center")

# plt.title("Spot–spot cell-cell communication network")
print("Cell–cell communication network")
plt.axis("off")
plt.tight_layout()
plt.savefig(
    fig_dir+"/52.Network.Auco.SC_CCC.png",
    format="png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## Auco SC Cell cluster_annos communication

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 读取数据（行列都是 cell type）
df = pd.read_csv("/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_MDIC3/Auco_SC/celltype_communication.txt", sep="\t", index_col=0)

# 2. 计算统计量
vmin = df.values.min()
vmax = df.values.max()
vcenter = df.values.mean()

print(f"min={vmin:.3f}, mean={vcenter:.3f}, max={vmax:.3f}")

# 3. 画热图
plt.figure(figsize=(7, 6.5))

sns.heatmap(
    df,
    cmap="bwr",                 # blue-white-red
    vmin=vmin,
    vmax=vmax,
    center=vcenter,             # 关键：以均值为中心
    square=False,
    linewidths=0,
    cbar_kws={
        "ticks": [vmin, vcenter, vmax],
        "label": "CCC strength"
    }
)

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
# 4. 保存
plt.savefig(
    fig_dir+"/52.Heatmap.Auco.SC_celltype_CCC.pdf",
    format="pdf",
    dpi=300,
    bbox_inches="tight"
)
plt.show()